In [1]:
import json, joblib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
import constraints as C

P = Path("processed")
df = pd.read_csv("../data2/IoT Network Intrusion Dataset.csv")
LAB = ["Label", "Cat", "Sub_Cat"]
ID_COLS = ["Flow_ID", "Src_IP", "Dst_IP", "Timestamp", "Src_Port"]

# identical cleaning to Phase 1
df = df.replace([np.inf, -np.inf], np.nan).dropna().drop(columns=ID_COLS)
feat = [c for c in df.columns if c not in LAB]
df = df.drop(columns=[c for c in feat if df[c].nunique() <= 1]).drop_duplicates().reset_index(drop=True)
y = (df["Label"] == "Anomaly").astype(int); X = df.drop(columns=LAB)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val_raw, X_te_raw, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)
X_val_raw = X_val_raw.reset_index(drop=True); y_val = y_val.reset_index(drop=True).values

meta = json.load(open(P / "meta.json")); F, LOG = meta["features"], meta["log_cols"]
scaler = joblib.load(P / "scaler.joblib")
assert F == list(X.columns)

# sanity: rebuilt raw split must reproduce the saved scaled array
X_val_s = C.to_scaled(X_val_raw, scaler, LOG, F)
print("matches saved X_val.npy:", np.allclose(X_val_s, np.load(P / "X_val.npy"), atol=1e-9))

matches saved X_val.npy: True


In [2]:
g = C.groups(F)
print({k: len(v) for k, v in g.items()})
json.dump(g, open(P / "feature_groups.json", "w"), indent=1)
g

{'controllable': 12, 'derived': 21, 'fixed': 35}


{'controllable': ['Flow_Duration',
  'TotLen_Fwd_Pkts',
  'TotLen_Bwd_Pkts',
  'Fwd_Pkt_Len_Max',
  'Fwd_Pkt_Len_Std',
  'Bwd_Pkt_Len_Max',
  'Bwd_Pkt_Len_Std',
  'Flow_IAT_Max',
  'Fwd_IAT_Tot',
  'Fwd_IAT_Max',
  'Bwd_IAT_Tot',
  'Bwd_IAT_Max'],
 'derived': ['Fwd_Pkt_Len_Mean',
  'Bwd_Pkt_Len_Mean',
  'Flow_Byts/s',
  'Flow_Pkts/s',
  'Flow_IAT_Mean',
  'Flow_IAT_Std',
  'Fwd_IAT_Mean',
  'Fwd_IAT_Std',
  'Bwd_IAT_Mean',
  'Bwd_IAT_Std',
  'Fwd_Pkts/s',
  'Bwd_Pkts/s',
  'Pkt_Len_Max',
  'Pkt_Len_Mean',
  'Pkt_Len_Std',
  'Pkt_Len_Var',
  'Pkt_Size_Avg',
  'Fwd_Seg_Size_Avg',
  'Bwd_Seg_Size_Avg',
  'Subflow_Fwd_Byts',
  'Subflow_Bwd_Byts'],
 'fixed': ['Dst_Port',
  'Protocol',
  'Tot_Fwd_Pkts',
  'Tot_Bwd_Pkts',
  'Fwd_Pkt_Len_Min',
  'Bwd_Pkt_Len_Min',
  'Flow_IAT_Min',
  'Fwd_IAT_Min',
  'Bwd_IAT_Min',
  'Bwd_PSH_Flags',
  'Bwd_URG_Flags',
  'Fwd_Header_Len',
  'Bwd_Header_Len',
  'Pkt_Len_Min',
  'FIN_Flag_Cnt',
  'SYN_Flag_Cnt',
  'RST_Flag_Cnt',
  'PSH_Flag_Cnt',
  'ACK_Flag_Cn

In [3]:
t1 = C.check(X_val_raw)
print("rules:", len(t1), "| rules with violations:", int((t1.violation_rate > 0).sum()))
t1

rules: 25 | rules with violations: 0


,rule,rows_checked,violation_rate
0,no negative values (except Init_Bwd_Win_Byts),30067,0.0
1,integer-valued size/time features,30067,0.0
2,Subflow bytes == total bytes,30067,0.0
3,Seg_Size_Avg == packet-length mean,30067,0.0
4,Fwd mean == bytes/packets,20320,0.0
5,Bwd mean == bytes/packets,30067,0.0
6,Pkt_Len_Var == Pkt_Len_Std^2,30067,0.0
7,"Pkt_Len_Max == max(Fwd, Bwd max)",30067,0.0
8,Flow_Pkts/s == packets/duration,30067,0.0
9,Flow_Byts/s == bytes/duration,30067,0.0


In [4]:
R0 = C.repair(X_val_raw, X_val_raw)
print("max abs diff after repair(clean, clean):", float(np.abs(R0.values - X_val_raw.values).max()))
back = C.to_scaled(C.to_raw(X_val_s, scaler, LOG, F), scaler, LOG, F)
print("roundtrip max abs diff:", float(np.abs(back - X_val_s).max()))

max abs diff after repair(clean, clean): 2.9802322387695312e-08
roundtrip max abs diff: 1.1102230246251565e-16


In [5]:
rng = np.random.default_rng(0)
mask = C.controllable_mask(F)
for eps in [0.02, 0.05, 0.10]:
    adv = np.clip(X_val_s + rng.uniform(0, eps, X_val_s.shape) * mask, 0, 1)
    naive = C.check(C.to_raw(adv, scaler, LOG, F), X_val_raw)
    _, raw_rep = C.project(adv, X_val_raw, scaler, LOG, F)
    rep = C.check(raw_rep, X_val_raw)
    print(f"eps={eps}: rules violated naive {int((naive.violation_rate>0).sum())}/{len(naive)}"
          f"  | repaired {int((rep.violation_rate>0).sum())}/{len(rep)}")

eps=0.02: rules violated naive 17/27  | repaired 0/27
eps=0.05: rules violated naive 17/27  | repaired 0/27
eps=0.1: rules violated naive 17/27  | repaired 0/27


In [6]:
rows = []
for eps in [0.01, 0.02, 0.05, 0.10]:
    adv = np.clip(X_val_s + rng.uniform(0, eps, X_val_s.shape) * mask, 0, 1)
    _, r = C.project(adv, X_val_raw, scaler, LOG, F)
    row = {"eps": eps}
    for c in ["Flow_Duration", "TotLen_Fwd_Pkts", "TotLen_Bwd_Pkts", "Flow_IAT_Max"]:
        rel = (r[c] - X_val_raw[c]) / X_val_raw[c].replace(0, np.nan)
        row[c + "_median_%"] = round(float(rel.median() * 100), 1)
    rows.append(row)
pd.DataFrame(rows)

,eps,Flow_Duration_median_%,TotLen_Fwd_Pkts_median_%,TotLen_Bwd_Pkts_median_%,Flow_IAT_Max_median_%
0,0.01,7.5,0.7,0.0,7.3
1,0.02,15.9,1.4,0.0,15.7
2,0.05,44.9,3.1,0.0,44.6
3,0.10,113.0,5.1,0.0,110.8


In [7]:
tf, tb, atk = X_val_raw["Tot_Fwd_Pkts"], X_val_raw["Tot_Bwd_Pkts"], (y_val == 1)
print("attack flows: can pad fwd %.3f | pad bwd %.3f | pad either %.3f | median packets %d" %
      ((tf[atk] > 1).mean(), (tb[atk] > 1).mean(),
       ((tf[atk] > 1) | (tb[atk] > 1)).mean(), int((tf + tb)[atk].median())))

attack flows: can pad fwd 0.491 | pad bwd 0.335 | pad either 0.816 | median packets 3


In [8]:
def make_mlp(n=68):
    return nn.Sequential(nn.Linear(n,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(),
                         nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1))
net = make_mlp(); net.load_state_dict(torch.load(P / "models" / "mlp_none.pt")); net.eval()

def attack_recall(A):
    with torch.no_grad():
        p = torch.sigmoid(net(torch.tensor(A[atk], dtype=torch.float32)).squeeze(1)).numpy()
    return float((p >= 0.5).mean())

print("clean attack recall:", round(attack_recall(X_val_s), 4))
for eps in [0.02, 0.05, 0.10, 0.20]:
    adv = np.clip(X_val_s + rng.uniform(0, eps, X_val_s.shape) * mask, 0, 1)
    adv_p, _ = C.project(adv, X_val_raw, scaler, LOG, F)
    print(f"eps={eps}: attack recall after random constrained noise: {attack_recall(adv_p):.4f}")

clean attack recall: 0.9979
eps=0.02: attack recall after random constrained noise: 0.9918
eps=0.05: attack recall after random constrained noise: 0.9649
eps=0.1: attack recall after random constrained noise: 0.8770
eps=0.2: attack recall after random constrained noise: 0.6611


In [9]:
X_val_raw.to_pickle(P / "X_val_raw.pkl")